## Recommendations

## DukaSense: Recommendation Rules
All data is synthetic demo data
Analysis window: 2025-10-03 to 2025-12-31 (90 days).

Gross profit = revenue - cost of goods, before rent, wages, transport and electricity. It is never called net profit.

# The rules
## What DukaSense tells the owner 

1. Low stock - Stock is at or below the reorder level , its urgent if under 7 days of stock left, otherwise "soon"
2. Restock quantity - Only if the product is sold at least 5 units in 90 days ,enough for 14 days of sales, minus current stock 
3. Slow-moving - Fewer than 5 units in 90 days, consider a promotion or ordering less. If stock is above twice the reorder level, avoid restocking 
4. Strong seller - Top 20% of the shops products by 90-day gross profit, at least 5 units ,keep it well stocked
5. Low margin - The gross margin is below 15% ,we compare the buying and selling price
6. Margin watch - The gross margin is 15% - 20% | we watch the supplier price changes

Confidence:
HIGH = 20+ units and 4+ sales in 90 days
MEDIUM = 5 to 19 units. LOW = under 5 units
With too little sales history, no quantity is suggested

## What the owner sees 
- Low stock, urgent (SH001) - "Sukari 1kg: 1 left (reorder level 7), about 1 day of stock at recent sales. Check today's supplier price and consider restocking about 11-1 kg bag."
- Why: We only have 1 item left in stock, so we need to order more right away. We usually order more when we hit 7, and we've sold 75 items over the last 3 months, so the confidence is HIGH
- Low stock, no quantity (SH005): "Kahawa 100g: stock is at or below your reorder level, but it sold fewer than 5 units in 90 days. Decide whether you still want to stock it before reordering." Confidence LOW.
- Slow-moving (SH001) - "Sabuni ya kufulia 500g sold only 0 units in 90 days. You already hold more than double your reorder level, so avoid restocking." So the Confidence is LOW
- Strong seller (SH001) - "Sukari 1 kg is one of your top earners (KSh 1,972 estimated gross profit in 90 days). Keep it well stocked." The confidence is HIGH
- Low margin (SH005) - "Sukari 1kg earns only about 10% gross margin. Compare your buying price and selling price." The confidence is HIGH

Expected totals is 132 products, 12 shops; low stock is 39 (25 urgent,11 soon,3 no quantity), slow-moving 15, strong seller 24, low margin 7, margin watch 36

In [56]:
import pandas as pd
import math

PERIOD_START, PERIOD_END, DAYS = "2025-10-03", "2025-12-31", 90
SLOW_UNITS         = 5  
LOW_MARGIN         = 15.0
WATCH_MARGIN       = 20.0
STRONG_TOP_SHARE   = 0.20
STRONG_MIN_UNITS   = 5   
TARGET_COVER_DAYS  = 14   
URGENT_COVER_DAYS  = 7     
OVERSTOCK_X        = 2
LABEL = "SYNTHETIC DEMO DATA-rule-based,not a trained model"

In [15]:
FILE = r"C:\Users\murii\Downloads\DukaSense_Combined_Synthetic_Dataset_3.xls"

In [16]:
def load(path):
    d = pd.read_csv(path, low_memory=False)
    return (d[d.record_type == "SALE"].copy(),
            d[d.record_type == "PRODUCT_CATALOGUE"].copy(),
            d[d.record_type == "INVENTORY_SNAPSHOT"].copy())
FILE = r"C:\Users\murii\Downloads\DukaSense_Combined_Synthetic_Dataset_3.xls"
sales, catalogue, snapshot = load(FILE)
print(len(sales), "sale lines |", len(catalogue), "products |", len(snapshot), "stock snapshot rows")

13450 sale lines | 132 products | 132 stock snapshot rows


In [17]:
def product_table(sales, catalogue, snapshot):
    sales["dt"] = pd.to_datetime(sales["sale_date"])
    last90 = sales[(sales.dt >= PERIOD_START) & (sales.dt < "2026-01-01")]
    k = ["shop_id", "product_id"]
    recent = last90.groupby(k).agg(units_90=("quantity", "sum"),
                                   lines_90=("sale_line_id", "count"),
                                   gp_90=("line_estimated_gross_profit_kes", "sum")).reset_index()
    year = sales.groupby(k).agg(rev_y=("line_revenue_kes", "sum"),
                                gp_y=("line_estimated_gross_profit_kes", "sum")).reset_index()
    cat = catalogue[k + ["product_name", "product_unit", "unit_selling_price_kes", "unit_cost_kes"]]
    t = (cat.merge(snapshot[k + ["quantity_on_hand", "reorder_level"]], on=k)
            .merge(recent, how="left", on=k).merge(year, how="left", on=k))
    for c in ["units_90", "lines_90", "gp_90", "rev_y", "gp_y"]:
        t[c] = t[c].fillna(0)
    t["margin_pct"] = (t.gp_y / t.rev_y * 100).where(
        t.rev_y > 0, (t.unit_selling_price_kes - t.unit_cost_kes) / t.unit_selling_price_kes * 100).round(1)
    t["gp_rank"] = t.groupby("shop_id").gp_90.rank(pct=True, ascending=False)
    t["strong"] = (t.gp_rank <= STRONG_TOP_SHARE) & (t.units_90 >= STRONG_MIN_UNITS)
    return t

table = product_table(sales, catalogue, snapshot)
table[["shop_id", "product_name", "quantity_on_hand", "reorder_level", "units_90", "margin_pct"]].head(10)

,shop_id,product_name,quantity_on_hand,reorder_level,units_90,margin_pct
0,SH001,Unga wa mahindi 2 kg,7.0,7.0,71.0,21.0
1,SH001,Mchele 1 kg,7.0,6.0,46.0,20.3
2,SH001,Sukari 1 kg,1.0,7.0,75.0,15.7
3,SH001,Mafuta ya kupika 1 L,10.0,5.0,71.0,17.3
4,SH001,Maziwa 500 ml,11.0,7.0,75.0,21.0
5,SH001,Mkate,5.0,5.0,64.0,21.0
6,SH001,Mayai,5.0,18.0,65.0,25.9
7,SH001,Soda 500 ml,17.0,11.0,70.0,30.9
8,SH001,Dawa ya meno 100 ml,4.0,3.0,5.0,23.2
9,SH001,Betri AA,5.0,3.0,5.0,27.6


## The rules

In [24]:
#confidence level
def confidence(units, lines):
    if units >= 20 and lines >= 4: return "HIGH"
    if units >= 5: return "MEDIUM"
    return "LOW"

In [35]:
for name in ["rule_low_stock", "rule_slow_moving", "rule_strong_seller", "rule_margin", "confidence"]:
    print(name, "->", "OK" if name in globals() else "MISSING")

rule_low_stock -> OK
rule_slow_moving -> MISSING
rule_strong_seller -> MISSING
rule_margin -> OK
confidence -> OK


In [36]:
#low stock
def rule_low_stock(r):
    out = []
    if r.quantity_on_hand <= r.reorder_level:
        if r.units_90 >= SLOW_UNITS:
            daily = r.units_90 / DAYS
            cover = r.quantity_on_hand / daily
            qty = int(max(0, math.ceil(daily * TARGET_COVER_DAYS) - r.quantity_on_hand))
            level = "URGENT" if cover < URGENT_COVER_DAYS else "SOON"
            out.append((f"LOW_STOCK_{level}",
                f"{r.product_name}: {r.quantity_on_hand:g} left (reorder level {r.reorder_level:g}), "
                f"about {cover:.0f} day(s) of stock at recent sales. Check today's supplier price "
                f"and consider restocking about {qty} {r.product_unit}.",
                f"Stock {r.quantity_on_hand:g} <= reorder level {r.reorder_level:g}; sold "
                f"{r.units_90:g} in last 90 days; target {TARGET_COVER_DAYS} days cover."))
        else:
            out.append(("LOW_STOCK_NO_QTY",
                f"{r.product_name}: stock is at or below your reorder level, but it sold fewer than "
                f"{SLOW_UNITS} units in 90 days. Decide whether you still want to stock it before reordering.",
                "Below reorder level but not enough sales history to suggest a quantity."))
    return out

In [50]:
#slow-moving
def rule_slow_moving(r):
    out = []
    if r.units_90 < SLOW_UNITS:
        extra = (" You already hold more than double your reorder level, so avoid restocking."
                 if r.quantity_on_hand > OVERSTOCK_X * r.reorder_level else "")
        out.append(("SLOW_MOVING",
            f"{r.product_name} sold only {r.units_90:g} units in 90 days.{extra} "
            f"Consider a small promotion or ordering less.",
            f"Demo threshold: fewer than {SLOW_UNITS} units from {PERIOD_START} to {PERIOD_END}."))
    return out

In [49]:
#strong-seller
def rule_strong_seller(r):
    out = []
    if r.strong:
        out.append(("STRONG_SELLER",
            f"{r.product_name} is one of your top earners (KSh {r.gp_90:,.0f} estimated gross profit "
            f"in 90 days). Keep it well stocked.",
            f"Top {int(STRONG_TOP_SHARE*100)}% of this shop's products by 90-day gross profit, "
            f"min {STRONG_MIN_UNITS} units."))
    return out

In [48]:
#low margin and margin watch
def rule_margin(r):
    out = []
    if r.margin_pct < LOW_MARGIN:
        out.append(("LOW_MARGIN",
            f"{r.product_name} earns only about {r.margin_pct:.0f}% gross margin. "
            f"Compare your buying price and selling price.",
            f"Gross margin {r.margin_pct:.1f}% < {LOW_MARGIN:.0f}%. "
            f"Gross profit is before rent, transport, wages."))
    elif r.margin_pct < WATCH_MARGIN:
        out.append(("MARGIN_WATCH",
            f"{r.product_name} margin is {r.margin_pct:.0f}%, close to the low-margin line. "
            f"Keep an eye on supplier price changes.",
            f"Gross margin {r.margin_pct:.1f}% between {LOW_MARGIN:.0f}% and {WATCH_MARGIN:.0f}%."))
    return out

In [51]:
def recommend(r):
    return rule_low_stock(r) + rule_slow_moving(r) + rule_strong_seller(r) + rule_margin(r)

In [52]:
for name in ["rule_low_stock", "rule_slow_moving", "rule_strong_seller", "rule_margin", "confidence"]:
    print(name, "->", "OK" if name in globals() else "MISSING, run its cell")

rule_low_stock -> OK
rule_slow_moving -> OK
rule_strong_seller -> OK
rule_margin -> OK
confidence -> OK


**Trying one product**

In [53]:
row = table.iloc[2]
print(row.product_name, "| stock:", row.quantity_on_hand, "| sold in 90 days:", row.units_90)
for rule, message, why in recommend(row):
    print("\n", rule)
    print("  What to tell the owner:", message)
    print("  Show me why:", why)

Sukari 1 kg | stock: 1.0 | sold in 90 days: 75.0

 LOW_STOCK_URGENT
  What to tell the owner: Sukari 1 kg: 1 left (reorder level 7), about 1 day(s) of stock at recent sales. Check today's supplier price and consider restocking about 11 1 kg bag.
  Show me why: Stock 1 <= reorder level 7; sold 75 in last 90 days; target 14 days cover.

 STRONG_SELLER
  What to tell the owner: Sukari 1 kg is one of your top earners (KSh 1,972 estimated gross profit in 90 days). Keep it well stocked.
  Show me why: Top 20% of this shop's products by 90-day gross profit, min 5 units.

 MARGIN_WATCH
  What to tell the owner: Sukari 1 kg margin is 16%, close to the low-margin line. Keep an eye on supplier price changes.
  Show me why: Gross margin 15.7% between 15% and 20%.


**Run for all shops**

In [54]:
def run(path):
    t = product_table(*load(path))
    rows = []
    for r in t.itertuples():
        for code, msg, why in recommend(r):
            rows.append(dict(shop_id=r.shop_id, product_id=r.product_id, product_name=r.product_name,
                             rule_code=code, message=msg, why=why,
                             confidence=confidence(r.units_90, r.lines_90),
                             quantity_on_hand=r.quantity_on_hand, reorder_level=r.reorder_level,
                             units_sold_90d=r.units_90, gross_margin_pct=r.margin_pct, data_label=LABEL))
    return t, pd.DataFrame(rows)

table, recs = run(FILE)
recs.to_csv("expected_recommendations.csv", index=False)
recs.rule_code.value_counts()

rule_code
MARGIN_WATCH        36
LOW_STOCK_URGENT    25
STRONG_SELLER       24
SLOW_MOVING         15
LOW_STOCK_SOON      11
LOW_MARGIN           7
LOW_STOCK_NO_QTY     3
Name: count, dtype: int64

**Sanity check**

In [55]:
low = recs[recs.rule_code.str.startswith("LOW_STOCK")]
n_reorder = int((snapshot.stock_status == "REORDER").sum())
print("Low-stock recommendations:", len(low))
print("REORDER in snapshot:      ", n_reorder)
print("Match:",len(low) == n_reorder)

Low-stock recommendations: 39
REORDER in snapshot:       39
Match: True


**One shop's recommendations**

In [57]:
recs[recs.shop_id == "SH001"][["product_name", "rule_code", "confidence", "message"]]

,product_name,rule_code,confidence,message
0,Unga wa mahindi 2 kg,LOW_STOCK_SOON,HIGH,Unga wa mahindi 2 kg: 7 left (reorder level 7)...
1,Sukari 1 kg,LOW_STOCK_URGENT,HIGH,"Sukari 1 kg: 1 left (reorder level 7), about 1..."
2,Sukari 1 kg,STRONG_SELLER,HIGH,"Sukari 1 kg is one of your top earners (KSh 1,..."
3,Sukari 1 kg,MARGIN_WATCH,HIGH,"Sukari 1 kg margin is 16%, close to the low-ma..."
4,Mafuta ya kupika 1 L,STRONG_SELLER,HIGH,Mafuta ya kupika 1 L is one of your top earner...
5,Mafuta ya kupika 1 L,MARGIN_WATCH,HIGH,"Mafuta ya kupika 1 L margin is 17%, close to t..."
6,Mkate,LOW_STOCK_SOON,HIGH,"Mkate: 5 left (reorder level 5), about 7 day(s..."
7,Mayai,LOW_STOCK_URGENT,HIGH,"Mayai: 5 left (reorder level 18), about 7 day(..."
8,Sabuni ya kufulia 500 g,SLOW_MOVING,LOW,Sabuni ya kufulia 500 g sold only 0 units in 9...
